# SGLang vs vLLM —— 同硬件同模型受控对照

在**同一张 Colab T4**、**同一个 Qwen2.5-0.5B-Instruct**、**同一套压测脚本**下比较两个推理框架。
两边都暴露 OpenAI 兼容的 `/v1/chat/completions`，客户端一行不用改，
差异可归因到服务端实现。

| | vLLM | SGLang |
|---|---|---|
| 前缀复用 | `--enable-prefix-caching` | **RadixAttention**（默认开，`--disable-radix-cache` 关） |
| 批处理 | Continuous Batching | Continuous Batching |

**四组配置**：{vLLM, SGLang} × {前缀复用 开, 关}。指标：吞吐 tok/s、TTFT p50/p99、TPOT p50。

## 关于「上一轮为什么起不来」——之前的判断是错的

第一次尝试失败后我写过一句「因为 flashinfer 的采样 kernel 要运行时 JIT 编译、需要 CUDA toolkit」。
**2026-09-05 逐条实测证明这个诊断不成立。** `flashinfer-python` 作为 `sglang[srt]` 的依赖
照样被装进环境，而真正的四个阻塞没有一个和它有关（见第 1 节表格：
`kernels` × `huggingface_hub` 严格 dataclass、`torchaudio` 与 `torch` 版本错配、
transformers 4.x 缺 `PreTrainedConfig`、transformers 5.16 撞 `qwen3_asr`）。

留着这段是为了记住：**当时那句话是从症状猜的因果，没有验证过。**

## 两个后端参数

仍然显式指定，但理由变了——不是"绕开 flashinfer 的编译"，而是
**在 sm_75 上选一条不依赖预编译核、可复现的路径**：

| 组件 | 默认 | 本轮 | 已在第 2 节 `--help` 验证 |
|---|---|---|---|
| Attention | flashinfer | `--attention-backend triton` | 取值列表含 `triton` |
| Sampling  | flashinfer | `--sampling-backend pytorch` | 取值列表含 `pytorch` |

> 第 2 节先跑 `--help` 把真实存在的参数与合法取值列出来再用。
> 2026-09-05 那次它第一时间报了「不存在」，把我从"按记忆写参数直接起服务"拦下来了——
> 而真正的问题在依赖层，不在参数名。**这一格不要删。**

**这个 notebook 必须单独开一个 Colab 会话**，不要和 vLLM 那份共用运行时。


## 0. 环境探测（主进程全程不 import torch）

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout
print(out)
name = out.strip().splitlines()[-1].split(",")[0].strip()
print("显卡:", name)
print("Tensor Core:", "无（GTX 16 系）" if "GTX 16" in name else "有")
print()
print("注意：SGLang 在 sm_75 上不支持 bfloat16，会自动回落 float16。")
print("     本仓已定位 fp16 GEMM 在【无】Tensor Core 卡上的塌陷；T4 有 TC，不受影响。")


## 1. 装 SGLang —— 含 2026-09-05 实测出的四个环境阻塞的修法

装 `sglang[srt]`（服务端运行时），**不是** `sglang[all]`。

下面四条不是预防性的，是 2026-09-05 在 Colab T4 上一条条撞出来再解掉的。
每条都记了报错原文，别当成可省的保险措施删掉：

| # | 症状（报错原文） | 根因 | 修法 |
|---|---|---|---|
| 1 | `StrictDataclassFieldValidationError`，栈里是 `kernels/deps.py` → `huggingface_hub/dataclasses.py` | `kernels 0.14.1` 调 `PythonPackage(...)` 撞上 `huggingface_hub 0.36.2` 的严格 dataclass 校验；transformers 的 `integrations/hub_kernels.py` 会 import 它 | **卸掉 `kernels`**。它是 HF 的可选加速内核包，缺了 transformers 自动跳过 |
| 2 | `_patch_image_processor_kwargs` → `image_processing_utils` → `audio_utils` → `import torchaudio` → `_extension` 检查失败 | `torchaudio 2.11.0+cu128` 与 `torch 2.13.0` 不匹配；transformers 5.x 的 `audio_utils` 会 import 它 | **卸掉 `torchaudio`**，并且**一开始就别装**——SGLang 不需要它 |
| 3 | `ImportError: cannot import name 'PreTrainedConfig'` | `PreTrainedConfig` 是 transformers **5.x** 的名字（4.x 叫 `PretrainedConfig`）。SGLang 0.5.19 要 5.x，**不能降到 4.x** | 留在 5.x |
| 4 | `ValueError: 'qwen3_asr' is already used by a Transformers config` | transformers **5.16.1** 内置了 `qwen3_asr`，与 SGLang 自己的注册撞名 | **钉 `transformers==5.12.1`** |

> 走过的弯路，记下来免得重犯：我最初判断阻塞是「flashinfer 的采样 kernel 要 JIT 编译」，
> **这个诊断是错的**——`flashinfer-python` 作为依赖装进来了也没关系，四个阻塞没有一个和它有关。
> 中途还把 transformers 降到 4.57.6，反而触发了第 3 条。

判断条件是「三个包**全部**在场才跳过」，不是「有任意一个就跳过」。


In [ ]:
import importlib.metadata as md_, subprocess, sys, os, shutil

CHECK = ["sglang", "aiohttp", "torchvision"]      # 注意：不含 torchaudio，见上表第 2 条
TF_PIN = "transformers==5.12.1"                    # 见上表第 3、4 条

def ver(p):
    try:
        return md_.version(p)
    except Exception:
        return None

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=isinstance(cmd, str),
                          capture_output=True, text=True, **kw)

missing = [p for p in CHECK if ver(p) is None]
print("缺失:", missing or "无")

if missing:
    if shutil.which("cargo") is None:
        print("装 Rust 工具链（outlines_core 可能需要源码编译）...")
        sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
    print("cargo:", (sh(["cargo", "--version"]).stdout or "未找到").strip())
    print()
    print("装 sglang[srt] + aiohttp + torchvision + " + TF_PIN + "（约 5-10 分钟）...")
    r = sh([sys.executable, "-m", "pip", "install", "-q",
            "sglang[srt]", "aiohttp", "torchvision", TF_PIN])
    print("退出码:", r.returncode)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
else:
    print("三个包都在，跳过安装。")

# 无论装没装，这两条每次都要执行：sglang[srt] 会把 kernels 重新拉回来
print()
print("清掉两个会炸导入链的包（见上表 1、2 条）...")
for pkg in ["kernels", "torchaudio"]:
    u = sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg])
    print("  卸 %-12s 退出码 %s" % (pkg, u.returncode))

# transformers 必须是 5.12.1：5.16 撞 qwen3_asr，4.x 缺 PreTrainedConfig
if ver("transformers") != "5.12.1":
    print("transformers 现为", ver("transformers"), "→ 钉到 5.12.1")
    sh([sys.executable, "-m", "pip", "install", "-q", TF_PIN])

print()
print("=== 版本 ===")
for p in CHECK + ["transformers", "torch", "flashinfer-python"]:
    print("  %-20s %s" % (p, ver(p) or "（未安装）"))
print("  %-20s %s" % ("kernels", ver("kernels") or "已卸载 ✓"))
print("  %-20s %s" % ("torchaudio", ver("torchaudio") or "已卸载 ✓"))


## 2. 探测 flag —— 先确认参数真实存在，再去用

不凭记忆写参数。这一格把 `--help` 里所有 `*-backend` 参数列出来，
并逐个检查我们要用的两个在不在、合法取值是什么。

**若这里显示「不存在」，不要往下跑**——把输出发我，我按当前版本的真实参数名改。


In [ ]:
import subprocess, sys, re

r = subprocess.run([sys.executable, "-m", "sglang.launch_server", "--help"],
                   capture_output=True, text=True)
help_txt = r.stdout + r.stderr

if not help_txt.strip():
    print("调 --help 没有任何输出，说明 SGLang 没装好。返回码:", r.returncode)
else:
    backends = sorted(set(re.findall(r"--[a-z0-9-]*backend[a-z0-9-]*", help_txt)))
    print("=== 本版本里存在的 *-backend 参数 ===")
    for b in backends:
        print("  ", b)

    print()
    print("=== 本轮要用的两个 ===")
    ok = True
    for flag in ["--attention-backend", "--sampling-backend"]:
        hit = flag in help_txt
        print("  %-24s %s" % (flag, "存在 OK" if hit else "不存在 !!"))
        ok = ok and hit

    for flag in ["--attention-backend", "--sampling-backend"]:
        m = re.search(re.escape(flag) + r"[^\n]*?\{([^}]*)\}", help_txt)
        if m:
            print("  %s 可选值: %s" % (flag, m.group(1)))

    print()
    print("结论:", "可以继续往下跑。" if ok else "有参数对不上，先停，把上面这段发我。")


## 3. 写出压测脚本（与 vLLM 那轮同一份，客户端一行不改）

In [ ]:
import io
files = {}
files["bench_serving.py"] = '# -*- coding: utf-8 -*-\n"""vLLM 服务端压测：并发扫描下的吞吐 / TTFT / TPOT，以及前缀复用的效果。\n\n指标定义（与 JD 里那套一致）：\n  TTFT  Time To First Token   —— 首 token 延迟，决定交互体感\n  TPOT  Time Per Output Token —— 首 token 之后的平均出词间隔\n  吞吐   总输出 token 数 / 墙钟时间\n\n用法（先 bash serve.sh 起服务）：\n  python bench_serving.py                 # 并发扫描\n  python bench_serving.py --prefix-test   # 前缀复用对照\n"""\nimport argparse, asyncio, json, statistics as st, time\nimport aiohttp\n\nURL = "http://127.0.0.1:8000/v1/chat/completions"\nMODEL = "Qwen/Qwen2.5-0.5B-Instruct"\n\n# 一段较长的共享 system prompt：开 --enable-prefix-caching 后其 prefill 只算一次\nSHARED_PREFIX = (\n    "You are a meticulous technical assistant. Answer concisely and precisely. "\n    "Always reason step by step before answering. " * 20\n)\n\n\nasync def one_request(sess, prompt, max_tokens, use_prefix):\n    msgs = ([{"role": "system", "content": SHARED_PREFIX}] if use_prefix else []) + \\\n           [{"role": "user", "content": prompt}]\n    body = {"model": MODEL, "messages": msgs, "max_tokens": max_tokens,\n            "temperature": 0.0, "stream": True}\n    t0 = time.perf_counter()\n    ttft, n_tok, last = None, 0, t0\n    async with sess.post(URL, json=body) as resp:\n        async for raw in resp.content:\n            line = raw.decode("utf-8").strip()\n            if not line.startswith("data: ") or line == "data: [DONE]":\n                continue\n            delta = json.loads(line[6:])["choices"][0].get("delta", {})\n            if delta.get("content"):\n                now = time.perf_counter()\n                if ttft is None:\n                    ttft = now - t0\n                n_tok += 1\n                last = now\n    return dict(ttft=ttft or 0.0, total=last - t0, n_tok=n_tok)\n\n\nasync def run_batch(n_conc, n_req, max_tokens, use_prefix):\n    prompts = [f"Explain concept #{i} in distributed systems." for i in range(n_req)]\n    sem = asyncio.Semaphore(n_conc)\n\n    async def guarded(sess, p):\n        async with sem:\n            return await one_request(sess, p, max_tokens, use_prefix)\n\n    timeout = aiohttp.ClientTimeout(total=600)\n    async with aiohttp.ClientSession(timeout=timeout) as sess:\n        await one_request(sess, "warmup", 4, use_prefix)          # 预热\n        t0 = time.perf_counter()\n        rs = await asyncio.gather(*(guarded(sess, p) for p in prompts))\n        wall = time.perf_counter() - t0\n\n    tot_tok = sum(r["n_tok"] for r in rs)\n    tpots = [(r["total"] - r["ttft"]) / max(r["n_tok"] - 1, 1) for r in rs if r["n_tok"] > 1]\n    return dict(conc=n_conc, wall=wall, tput=tot_tok / wall, rps=len(rs) / wall,\n                ttft_p50=st.median(r["ttft"] for r in rs),\n                ttft_p99=sorted(r["ttft"] for r in rs)[int(len(rs) * 0.99) - 1],\n                tpot_p50=st.median(tpots) if tpots else 0.0, tot_tok=tot_tok)\n\n\nasync def sweep(args):\n    print(f"{\'并发\':>5}{\'请求\':>6}{\'墙钟s\':>9}{\'吞吐 tok/s\':>13}{\'RPS\':>8}"\n          f"{\'TTFT p50\':>11}{\'TTFT p99\':>11}{\'TPOT p50\':>11}")\n    print("-" * 74)\n    out = []\n    for c in [1, 2, 4, 8, 16, 32]:\n        r = await run_batch(c, max(c * 4, 16), args.max_tokens, use_prefix=False)\n        print(f"{r[\'conc\']:>5}{max(c*4,16):>6}{r[\'wall\']:>9.2f}{r[\'tput\']:>13.1f}"\n              f"{r[\'rps\']:>8.2f}{r[\'ttft_p50\']*1e3:>10.1f}ms{r[\'ttft_p99\']*1e3:>10.1f}ms"\n              f"{r[\'tpot_p50\']*1e3:>10.2f}ms")\n        out.append(r)\n    json.dump(out, open("sweep_results.json", "w"), indent=1)\n    base = out[0]["tput"]\n    print(f"\\ncontinuous batching 收益：并发 1 → 32，吞吐 "\n          f"{base:.1f} → {out[-1][\'tput\']:.1f} tok/s（{out[-1][\'tput\']/base:.1f}×），"\n          f"TTFT p50 {out[0][\'ttft_p50\']*1e3:.0f} → {out[-1][\'ttft_p50\']*1e3:.0f} ms")\n    print("吞吐与延迟的取舍就在这张表里：并发拉高吞吐涨，但 TTFT 同步恶化。")\n\n\nasync def prefix_test(args):\n    print("前缀复用对照（服务端需带 --enable-prefix-caching 启动）")\n    print(f"{\'场景\':<26}{\'吞吐 tok/s\':>13}{\'TTFT p50\':>12}")\n    print("-" * 51)\n    for label, up in [("无共享前缀", False), (f"共享前缀 ({len(SHARED_PREFIX)} 字符)", True)]:\n        r = await run_batch(8, 32, args.max_tokens, use_prefix=up)\n        print(f"{label:<26}{r[\'tput\']:>13.1f}{r[\'ttft_p50\']*1e3:>11.1f}ms")\n    print("\\n共享前缀命中 KV cache 后，重复的 prefill 不再重算，TTFT 应显著下降。")\n    print("对比未开 --enable-prefix-caching 重启服务再跑一次，差值即为该特性的真实收益。")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--max-tokens", type=int, default=128)\n    ap.add_argument("--prefix-test", action="store_true")\n    a = ap.parse_args()\n    asyncio.run(prefix_test(a) if a.prefix_test else sweep(a))\n'
files["isolate.py"] = '# -*- coding: utf-8 -*-\n"""隔离测试：一次性发 N 条并发请求，期间无新请求到达。\n用于区分「稳态 batch=N 解码慢」与「新请求 prefill 插队拖累解码」。"""\nimport asyncio, sys, time, json, statistics as st\nimport aiohttp\nURL="http://127.0.0.1:8000/v1/chat/completions"; MODEL="Qwen/Qwen2.5-0.5B-Instruct"\n\nasync def one(s, i, n_tok):\n    b={"model":MODEL,"messages":[{"role":"user","content":f"Explain idea {i} briefly."}],\n       "max_tokens":n_tok,"temperature":0.0,"stream":True}\n    t0=time.perf_counter(); ttft=None; n=0; last=t0\n    async with s.post(URL,json=b) as r:\n        async for raw in r.content:\n            l=raw.decode().strip()\n            if not l.startswith("data: ") or l=="data: [DONE]": continue\n            d=json.loads(l[6:])["choices"][0].get("delta",{})\n            if d.get("content"):\n                now=time.perf_counter()\n                if ttft is None: ttft=now-t0\n                n+=1; last=now\n    return (ttft or 0), last-t0, n\n\nasync def burst(n, n_tok=64):\n    """严格同时发 n 条，全部跑完才结束 —— 稳态就是 batch=n。"""\n    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=600)) as s:\n        await one(s,-1,4)                       # 预热\n        t0=time.perf_counter()\n        rs=await asyncio.gather(*(one(s,i,n_tok) for i in range(n)))\n        wall=time.perf_counter()-t0\n    tp=[(tot-tt)/max(k-1,1) for tt,tot,k in rs if k>1]\n    tot=sum(k for _,_,k in rs)\n    print(f"  一次性 {n:>2} 条并发: 墙钟 {wall:>6.2f}s  总吞吐 {tot/wall:>6.1f} tok/s  "\n          f"TPOT p50 {st.median(tp)*1e3:>7.2f} ms")\n\nasync def main():\n    print("=== 无新到达的纯稳态测试 ===")\n    for n in [1, 2, 3, 4, 8, 16]:\n        await burst(n)\n\nasyncio.run(main())\n'
for n, s in files.items():
    io.open(n, "w", encoding="utf-8").write(s)
    print("写出", n, len(s), "字符")


## 4. 启动器

起不来就把失败原因原样打出来，不静默重试。
`--mem-fraction-static` 在 T4（16 GiB）给 0.80；本地 4 GiB 卡当时只能给 0.55。


In [ ]:
import subprocess, sys, time, requests

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def serve_sglang(extra, tag, wait=300):
    subprocess.run(["pkill", "-f", "sglang.launch_server"], check=False)
    time.sleep(8)
    cmd = [sys.executable, "-m", "sglang.launch_server",
           "--model-path", MODEL,
           "--host", "127.0.0.1", "--port", "8000",
           "--context-length", "2048",
           "--mem-fraction-static", "0.80",
           "--attention-backend", "triton",    # 绕开 flashinfer 的注意力核
           "--sampling-backend", "pytorch",    # 绕开 flashinfer 的采样核（上轮死在这）
           ] + extra
    print("启动参数:", " ".join(cmd[2:]))
    print()
    log = open("/content/sgl_%s.log" % tag, "w")
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

    for i in range(wait // 2):
        if p.poll() is not None:
            print("[%s] 进程已退出（码 %s），日志尾部：" % (tag, p.returncode))
            print(open("/content/sgl_%s.log" % tag).read()[-4000:])
            return None
        try:
            if requests.get("http://127.0.0.1:8000/v1/models", timeout=2).status_code == 200:
                print("[%s] 就绪，用时 %ds" % (tag, i * 2))
                txt = open("/content/sgl_%s.log" % tag).read()
                for pat in ["KV Cache is allocated", "max_total_num_tokens",
                            "Use float16", "attention backend", "sampling backend"]:
                    for ln in txt.splitlines():
                        if pat.lower() in ln.lower():
                            print("   ", ln.strip()[:170])
                            break
                return p
        except requests.RequestException:
            pass          # 只吞连接失败；其它异常照抛，别再无声吞掉真 bug
        time.sleep(2)

    print("[%s] %ds 内没起来，日志尾部：" % (tag, wait))
    print(open("/content/sgl_%s.log" % tag).read()[-4000:])
    return None


## 5. SGLang · RadixAttention 开（默认）

RadixAttention 是 SGLang 的前缀复用机制，默认开启，对应 vLLM 的 `--enable-prefix-caching`。


In [ ]:
proc = serve_sglang([], "radix_on")

In [ ]:
!python -u isolate.py

In [ ]:
!python -u bench_serving.py

In [ ]:
!python -u bench_serving.py --prefix-test

## 6. SGLang · RadixAttention 关（受控对照）

In [ ]:
proc = serve_sglang(["--disable-radix-cache"], "radix_off")

In [ ]:
!python -u bench_serving.py --prefix-test

## 7. 怎么读这批数

把四组数并排（vLLM 那两组在 `cloud_vllm_verify.ipynb` 里已经跑过）：

| | 前缀复用 开 | 前缀复用 关 | 差值 |
|---|---|---|---|
| vLLM（`--enable-prefix-caching`） | | | |
| SGLang（RadixAttention） | | | |

两个维度：

1. **横向** = 各框架前缀复用的真实收益
2. **纵向** = 同一开关状态下两框架的实现差异

### 边界（写进结论，一条都不许省）

- 两边都是 fp16（sm_75 不支持 bf16，SGLang 自动回落），精度口径一致，可比。
- SGLang 这里跑的是 **Triton attention + PyTorch sampling**，不是它的默认最优路径
  （默认走 flashinfer）。**所以这组数字不能说成「SGLang 的最佳性能」**，
  只能说成「在不引入 CUDA toolkit 依赖的前提下，两个框架的可复现对照」。
  这条必须进 README——否则就是拿次优配置去黑一个框架。
- 0.5B 模型 prefill 本来就便宜，共享前缀只有约 2380 字符，
  前缀复用的收益天然被压缩。**差值很小就如实写，不要包装。**
